# Day 67：面试专题（一）—— 技术八股与模拟笔试 (`v1.0.0-day67`)

本 Notebook 演示《智枢企业级智能体平台》Day 67 核心模块 `zhishu_agent.interview_bank`：
1. 加载并筛选五大维度（`Transformer` / `RAG` / `Agent` / `Python` / `Engineering`）高频技术八股题库；
2. 使用 `grade_answer()` 对不同质量作答（高分、部分命中、空答）进行要点覆盖率评分；
3. 使用 `ExamSession` 运行 5 题跨维度模拟笔试，自动输出薄弱维度诊断、靶向复习计划与 Markdown 报告。

In [1]:
import sys
from pathlib import Path

# 自动定位 project_iterations/day_67
cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "project_iterations" / "day_67").exists()), None)
assert repo_root is not None, "未找到 project_iterations/day_67 目录"
day67_dir = repo_root / "project_iterations" / "day_67"
if str(day67_dir) not in sys.path:
    sys.path.insert(0, str(day67_dir))

from zhishu_agent.interview_bank import (
    VALID_CATEGORIES,
    ExamSession,
    get_question_bank,
    grade_answer,
    list_questions,
)

bank = get_question_bank()
print(f"题库加载成功，共 {len(bank)} 道题，覆盖维度: {VALID_CATEGORIES}")
for cat in VALID_CATEGORIES:
    print(f"  - {cat}: {len(list_questions(category=cat))} 题")

题库加载成功，共 20 道题，覆盖维度: ('Transformer', 'RAG', 'Agent', 'Python', 'Engineering')
  - Transformer: 4 题
  - RAG: 4 题
  - Agent: 4 题
  - Python: 4 题
  - Engineering: 4 题


## 1. 单题要点覆盖率评分演示（`grade_answer`）

In [2]:
q_tf = list_questions(category="Transformer")[0]
print("题目:", q_tf.question)

samples = {
    "高分回答": "因为点积方差随 d_k 线性增大，数值过大容易让 Softmax 进入梯度饱和区引发梯度消失，除以 sqrt(d_k) 可将方差恢复为 1，稳定反向传播。",
    "部分回答": "主要是为了防止 softmax 饱和导致梯度消失。",
    "空回答": "   ",
}

for label, ans in samples.items():
    res = grade_answer(q_tf, ans)
    print(f"\n[{label}] 得分: {res.score}/100 | 覆盖率: {res.coverage_ratio:.0%} | 空答: {res.is_empty}")
    print("  命中:", res.matched_points)
    print("  缺失:", res.missing_points)
    print("  评语:", res.feedback)

题目: 为什么 Self-Attention 在计算点积后要除以 sqrt(d_k)？

[高分回答] 得分: 100.0/100 | 覆盖率: 100% | 空答: False
  命中: ['点积方差随维度 d_k 线性增大', '防止点积结果过大导致 Softmax 进入梯度饱和区', '缩放后维持方差为 1，稳定反向传播训练']
  缺失: []
  评语: 回答全面且要点精准，已具备大厂技术面标准表达水准。

[部分回答] 得分: 40.5/100 | 覆盖率: 33% | 空答: False
  命中: ['防止点积结果过大导致 Softmax 进入梯度饱和区']
  缺失: ['点积方差随维度 d_k 线性增大', '缩放后维持方差为 1，稳定反向传播训练']
  评语: 核心得分点覆盖不足，建议对照参考答案重点补齐：点积方差随维度 d_k 线性增大；缩放后维持方差为 1，稳定反向传播训练。

[空回答] 得分: 0.0/100 | 覆盖率: 0% | 空答: True
  命中: []
  缺失: ['点积方差随维度 d_k 线性增大', '防止点积结果过大导致 Softmax 进入梯度饱和区', '缩放后维持方差为 1，稳定反向传播训练']
  评语: 未检测到有效作答内容。建议先梳理核心概念，再逐条补充关键机制与工程取舍。


## 2. 跨维度模拟笔试与薄弱维度诊断（`ExamSession`）

In [3]:
exam_qs = ExamSession.select_questions(n=5, seed=67)
session = ExamSession(questions=exam_qs, candidate_name="陈默", session_id="NB-DAY67-01")

# 模拟前 3 题高质量作答，第 4 题部分作答，第 5 题留空
for idx, q in enumerate(exam_qs):
    if idx < 3:
        ans = q.model_answer
    elif idx == 3:
        ans = q.key_points[0].keywords[0]
    else:
        ans = ""
    r = session.submit_answer(q.id, ans)
    print(f"Q{idx+1} [{q.category}] {q.id}: {r.score:.1f} 分")

report = session.finish_exam(weak_threshold=70.0)
print("\n综合均分:", report.average_score, "| 评级:", report.grade_level)
print("各维度得分:", report.category_scores)
print("薄弱维度:", report.weak_categories)
assert len(report.weak_categories) >= 1
assert report.to_dict()["total_questions"] == 5

Q1 [Python] PY-004: 100.0 分
Q2 [Agent] AGT-003: 69.2 分
Q3 [RAG] RAG-004: 100.0 分
Q4 [Engineering] ENG-004: 35.9 分
Q5 [Transformer] TF-003: 0.0 分

综合均分: 61.0 | 评级: C（及格边缘 · 需重点强化高频八股）
各维度得分: {'Python': 100.0, 'Agent': 69.2, 'RAG': 100.0, 'Engineering': 35.9, 'Transformer': 0.0}
薄弱维度: ['Transformer', 'Engineering', 'Agent']


## 3. 导出 Markdown 诊断报告

In [4]:
md_report = report.to_markdown()
print(md_report[:1200])

# 智枢模拟笔试诊断报告 (NB-DAY67-01)

- **候选人**: 陈默
- **作答进度**: 4/5
- **综合均分**: 61.0 / 100（评级: **C（及格边缘 · 需重点强化高频八股）**）
- **薄弱维度**: Transformer, Engineering, Agent

## 各维度得分

| 技术维度 | 平均分 | 状态 |
| :--- | :---: | :---: |
| Python | 100.0 | ✅ 达标 |
| Agent | 69.2 | ⚠️ 薄弱需加强 |
| RAG | 100.0 | ✅ 达标 |
| Engineering | 35.9 | ⚠️ 薄弱需加强 |
| Transformer | 0.0 | ⚠️ 薄弱需加强 |

## 逐题评分详情

### 1. [Python | Hard] Python GIL（全局解释器锁）对多线程并发有什么影响？AI 后端如何绕过 GIL 提升 CPU 密集型吞吐？（得分：100.0/100）
- **命中要点**: GIL 保证同一时刻只有一个线程执行 Python 字节码，限制 CPU 密集型多线程扩展, I/O 密集型任务（网络/磁盘）在等待时会释放 GIL，因此多线程/asyncio 依然高效, CPU 密集型可通过多进程（multiprocessing/Uvicorn workers）或 C/Rust 扩展（NumPy/PyTorch）绕过
- **缺失要点**: 无
- **评语**: 回答全面且要点精准，已具备大厂技术面标准表达水准。

### 2. [Agent | Hard] 企业级 Agent 如何实现 Human-in-the-Loop（HITL）的挂起与断点恢复？（得分：69.2/100）
- **命中要点**: 在敏感工具调用前拦截执行，将上下文与待审批动作持久化为 pending 状态, 释放当前请求线程/协程，通过审批回调或幂等接口触发后续恢复执行
- **缺失要点**: 校验审批人权限与审批状态流转（pending -> approved/rejected），防止越权或重放
- **评语**: 命中了核心方向，但仍缺少 1 个关键得分点：校验审批人权限与审批状态流转（pending -> approved/rejecte